# homeassistant / hotwater / temp

**Intent:** From the pipe temperature, the clock, the solar radiation, the booster's last command and an ambient temperature, with a model state carried between calls, output a state code, a flow flag, a cooled flag and the water temperature only while hot water is flowing. The signature is not yet defined: the scaffold names are placeholders (`homeassistant_hotwater_in` and `homeassistant_hotwater_out`).

**Notes:**
- Purpose: Report the hot water temperature only while hot water is flowing, say when the pipe has cooled with no flow, and not mistake booster or solar heating for flow
- Status: a placeholder, the sections below are the stages still to be built
- Data splits: `training` and `validation` readings of the pipe temperature with the booster commands and solar radiation; `backtest` whole recorded days; `golden` rows for cooled, flowing, heated with no flow, and a missing input

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** load `env` and declare the model name, version and dataset names as constants (`model.md` 6.6).

In [ ]:
from workbench.data import golden_data
from workbench.env import init_env
from workbench.project import load_signature, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "homeassistant/hotwater/temp"
MODEL_NAME = "homeassistant_hotwater_temp"
TARGETS = [{"kind": "served"}]
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
GOLDEN = "hotwater_temp_golden"
CONSTANTS = {"homeassistant_hotwater_out": 1}

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** load the `training` and `validation` data.
- **Blocked:** needs a read-only query of `influxdb3` for the pipe temperature, solar radiation and the booster commands from 2026-08-13, and a flow split that does not exist yet because there is no flow meter (`plan.md`, Open decisions).

## Explore

- **Stage:** 2, explore.
- **Intent:** plot and check the captured data; nothing reported from it ships (R4).
- **Blocked:** needs the captured data. Plot the rate of change of the pipe, mark the flow-like events and settle the flow split, the booster state and the ambient sensor.

## Fit

- **Stage:** 3, fit.
- **Intent:** fit the model on `training` only (R6).
- **Blocked:** needs the captured data to fit the state thresholds from `training`. The builder exists: write the state rules as one Python function and trace it with `workbench.wasm.trace` (`model.md` 4.4, `example/wasm/saturate`), with `select` for each branch.

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** check the fit on `validation` and `backtest` through the shipped runtime (R4).
- **Blocked:** needs a fitted `wasm` module, the shipped runtime over `validation` and `backtest` and an `acceptance_checks` function.

## Package

- **Stage:** 5, package.
- **Intent:** write the golden samples, build the signature and write the project (R1, R3).
- **Blocked:** the golden file exists but the signature is a stand-in. Replace it with the pipe temperature, clock, solar radiation, booster command and ambient inputs, the state vectors and the state code, flow flag, cooled flag and water temperature outputs, then call `package_project`.

In [ ]:
from workbench.model import register_preview

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {"homeassistant_hotwater_in": [1.0, None], "expect": ["ok", "error"]})

project_dir = scratch_project_dir(WORKSPACE)
print("no model is fitted yet, so nothing is packaged")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).
- **Blocked:** nothing to register until a project is packaged. `TARGETS` is a `served` target, and the cell below fails until then.

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS)
print("registered version", version)